# Level 7: Class-based and stateful decorators

## 7.1 A class as a decorator

Any object with a `__call__` method can be called like a function. So a class can be a decorator: `__init__` receives the function, `__call__` runs on every call. This is handy when the decorator needs to keep state.

In [ ]:
import functools
import time
import random

In [ ]:
class CountCalls:
    def __init__(self, func):
        functools.update_wrapper(self, func)   # the class version of @wraps
        self.func = func
        self.calls = 0

    def __call__(self, *args, **kwargs):
        self.calls += 1
        print(f"Call {self.calls} of {self.func.__name__}")
        return self.func(*args, **kwargs)

@CountCalls
def say(msg):
    return msg

say("a"); say("b"); say("c")
print("total:", say.calls)
print(say.__name__)

## 7.2 A class-based decorator with arguments

Now `__init__` receives the arguments and `__call__` receives the function.

In [ ]:
class Repeat:
    def __init__(self, times):
        self.times = times

    def __call__(self, func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            return [func(*args, **kwargs) for _ in range(self.times)]
        return wrapper

@Repeat(times=3)
def roll():
    import random
    return random.randint(1, 6)

print(roll())

## 7.3 Closure state with `nonlocal`

You do not always need a class. `nonlocal` lets the wrapper change a variable from the enclosing function.

In [ ]:
def call_limit(max_calls):
    def decorator(func):
        count = 0
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            nonlocal count           # without this, count += 1 raises UnboundLocalError
            if count >= max_calls:
                raise RuntimeError(f"{func.__name__} can only be called {max_calls} times")
            count += 1
            return func(*args, **kwargs)
        return wrapper
    return decorator

@call_limit(2)
def free_trial():
    return "feature used"

print(free_trial())
print(free_trial())
try:
    free_trial()
except RuntimeError as e:
    print("Error:", e)